<a href="https://colab.research.google.com/github/ironcevic/Modelling_low_dimensional_materials/blob/main/week2/polyacetylene.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Your first QE calculation

Let's first take a look at the files in this directory.

```cumulene.scf.in``` runs the SCF calculation and converges the energy and the density.

```cumulene.nscf.in``` runs an NSCF calculation on a denser k-grid.

```cumulene.dos.in``` is a post-processing calculation that will give us the density of states (DoS).

```cumulene.bands.in``` is a post-processing calculation that will give us the band structure.

We will start by inspecting the flags and the geometry in the ```cumulene.scf.in``` file.
Execute the code below, and then open the .xsf file in Vesta and the .xyz in Avogadro.

In [ ]:
from ase.io import read, write # install ASE via conda or pip first

atoms = read('cumulene.scf.in', format='espresso-in') # load geometry into an atoms object
write('unitcell.xsf', atoms) # write the atoms object as an xsf
supercell = atoms.repeat((10, 1, 1)) # create supercell object
write('supercell.xyz', supercell) # write supercell object as xyz

### 1.1. Pseudopotentials and the SCF calculation

Before running the calculation, you need the pseudopotentials (PPs).
To download PPs this, go to https://www.quantum-espresso.org/pseudopotentials/ and download KJ-PAW PBE potentials in .upf format.
Then run the code below.

In [ ]:
!pw.x < cumulene.scf.in > cumulene.scf.out # execute pw.x with cumulene.scf.in as input and cumulene.scf.out as output
!grep "JOB DONE" cumulene.scf.out # look for "JOB DONE" in the output file to check if the calculation finished successfully

We can parse the output by combining bash and Python code.

In [ ]:
e_fermi_cumulene = !grep "the Fermi energy is" cumulene.scf.out | awk '{print $5}' # write the Fermi energy into a python object
e_fermi_cumulene = float(e_fermi_cumulene[0]) # convert the Fermi energy from a string to a float

e_total_cumulene = ! grep "total energy       " cumulene.scf.out | tail -1 | awk '{print $5}' # write the total energy into a python object
e_total_cumulene = float(e_total_cumulene[0]) # convert the total energy from a string to a float
print(f"The Fermi energy is {e_fermi_cumulene} eV.")
print(f"The total energy is {e_total_cumulene} eV.")

Before plotting the DoS, let's load some libraries and a colour dictionary.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
mpl.rcParams['font.size'] = 16

colours = {
    "green": "#00B828",
    "yellow": "#FFD900",
    "purple": "#800FF2",
    "blue": "#0073FF",
    "orange": "#FF5000",
    "grey": "#B3B3B3",
}
plt.rcParams.update({
    'xtick.major.width': 2,     # x-tick thickness
    'ytick.major.width': 2,     # y-tick thickness
    'xtick.major.size': 5,        # x-tick length
    'ytick.major.size': 5,        # y-tick length
    'axes.linewidth': 2,         # Thickness of axis border (applies to spines)
    'lines.linewidth': 2
})

### 1.2. NSCF calculation

To get a denser **k**-grid, we will now run a non-self-consistent-field calculation.

In [ ]:
!pw.x < cumulene.nscf.in > cumulene.nscf.out # execute pw.x with cumulene.nscf.in as input and cumulene.nscf.out as output
!grep "JOB DONE" cumulene.nscf.out # look for "JOB DONE" in the output file to check if the calculation finished successfully

### 1.3. Density of states

Now we shall compute the density of states (DoS). Inspect ```cumulene.dos.in```. How many eV in each direction relative to ```e_fermi``` does it include?
Note that this is a post-processing calculation - it only evaluates the DoS.
Now run the code below. Which files does it produce?

In [ ]:
!dos.x < cumulene.dos.in > cumulene.dos.out # execute dos.x with cumulene.dos.in as input and cumulene.dos.out as output
!grep "JOB DONE" cumulene.dos.out # look for "JOB DONE" in the output file to check if the calculation finished successfully

Now we will create the dos object. Inspect ```cumulene.bands.in```.

In [ ]:
dos_cumulene = np.genfromtxt("cumulene.dos", skip_header=1) # create the dos object
print(np.shape(dos_cumulene)) # check the shape of the dos object

# try printing columns and rows of the dos


Now let's plot the DoS.

In [ ]:
energy_limits = [-8, 3]
e_fermi = e_fermi_cumulene
dos = dos_cumulene

# these two lines print the DOS and label the Fermi energy
plt.plot(dos[:, 0], dos[:, 1], color = "k") # plot the dos as a black line
plt.axvline(e_fermi, linestyle='dashed', color = "k") # fermi energy

# the next two lines make a fill with a colour depending on occupancy
plt.fill_between(dos[:, 0], dos[:, 1], where=(dos[:, 0] < e_fermi),
                 facecolor=colours["blue"], alpha=0.5, label='occupied')
plt.fill_between(dos[:, 0], dos[:, 1], where=(dos[:, 0] >= e_fermi),
                 facecolor=colours["orange"], alpha=0.5, label='unoccupied')

# various labels
plt.xlabel("energy (eV)")
plt.ylabel("density of states")
plt.xlim(energy_limits)
plt.ylim(0, 3)
plt.legend(loc="upper left", frameon=False, bbox_to_anchor=(0.05, 1))
plt.show()

### 1.4. Band structure

Now we shall work on the the band structure. Inspect ```cumulene.bands.in```, and then run the post-processing calculation. Which files does it produce?

In [ ]:
!bands.x < cumulene.bands.in > cumulene.bands.out
!grep "JOB DONE" cumulene.bands.out # look for "JOB DONE" in the output file to check if the calculation finished successfully

Let us now plot the band structure. Inspect the ```cumulene.bands.dat.gnu``` file, and then reate the ```bands``` object using the provided code.


In [ ]:
bands_cumulene = np.genfromtxt("cumulene.bands.dat.gnu") # create the bands object
print(np.shape(bands_cumulene)) # what is its shape? compare with the .gnu file
bands_cumulene = np.split(bands_cumulene, 9) # split it into 9 bands
print(np.shape(bands_cumulene)) # what is its shape now?
# now try printing the energies of the first band

In [ ]:
bands = bands_cumulene

# here below we shall plot the band structure
for band in bands:
    x = band[:, 0]
    y = band[:, 1]
    # can you figure out what the four lines below do?
    if np.all(y < e_fermi):
        plt.plot(x, y, color=colours["blue"])
    else:
        plt.plot(x, y, color=colours["orange"])
plt.ylim(energy_limits)
plt.xlim(0, 0.5)
plt.axhline(y=e_fermi, color='k', linestyle='--')
plt.xlabel(r"$k$")
plt.ylabel("energy (eV)")
plt.annotate("occupied", xy=(0.01, e_fermi-0.8), color=colours["blue"])
plt.annotate("unoccupied", xy=(0.01, e_fermi+0.5), color=colours["orange"])
plt.show()

For convenience, here is a single block that plots the band structure and the DoS.
Try to reformat it into a function – this will come in handy later.

In [ ]:
# these three lines define the parameters of the plot
file_name = "polyene"
e_range = 5 # how far away from the Fermi energy do we want to plot?
n_bands = 9 # how many bands are there in the .gnu file?

# make objects
with open(f"{file_name}.dos") as f:
    first_line = f.readline()
    e_fermi = float(first_line.split()[8])
energy_limits = [e_fermi - e_range, e_fermi + e_range]
dos = np.genfromtxt(file_name+".dos", skip_header=1) # check file name
bands = np.genfromtxt(file_name+".bands.dat.gnu") # check file name
bands = np.split(bands, n_bands) # split the bands object into separate bands
# plot band structure
fig, ax = plt.subplots(1,2, figsize=(10, 5), gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.1})
for band in bands:
    x = band[:, 0]
    y = band[:, 1]
    # can you figure out what the four lines below do?
    if np.all(y < e_fermi):
        ax[0].plot(x, y, color=colours["blue"])
    else:
        ax[0].plot(x, y, color=colours["orange"])
ax[0].set_ylim(energy_limits)
ax[0].set_xlim(0, 0.5)
ax[0].axhline(y=e_fermi, color='k', linestyle='--')
ax[0].set_xlabel(r"$k$")
ax[0].set_ylabel("energy (eV)")
ax[0].annotate("occupied", xy=(0.01, e_fermi-0.6), color=colours["blue"])
ax[0].annotate("unoccupied", xy=(0.01, e_fermi+0.25), color=colours["orange"])

# plot dos
ax[1].plot(dos[:, 1], dos[:, 0], color = "k") # plot the dos with swapped axes
ax[1].axhline(e_fermi, linestyle='dashed', color = "k") # fermi energy as horizontal line
ax[1].fill_betweenx(dos[:, 0], dos[:, 1], where=(dos[:, 0] < e_fermi),
                 facecolor=colours["blue"], alpha=0.5, label='occupied')
ax[1].fill_betweenx(dos[:, 0], dos[:, 1], where=(dos[:, 0] >= e_fermi),
                 facecolor=colours["orange"], alpha=0.5, label='unoccupied')
ax[1].set_ylim(energy_limits)
ax[1].set_xlim(0, 1.2 * max(dos[:, 1]))
ax[1].set_xlabel("density of states")
ax[1].tick_params(labelleft=False, left=False)
fig.savefig(f"{file_name}_bands_dos.pdf", bbox_inches='tight') # save the figure as a pdf
plt.show() # show the figure

print(np.max(bands[4][ :, 1]))
print(np.min(bands[5][ :, 1]))


### 1.5. Geometry relaxation

All these results were obtained for a cumulenic geometry.
Let's now relax the geometry and do it all again.
Inspect ```polyene.relax.in```. Which new keywords does it have? What do you expect to get?

In [ ]:
!pw.x < polyene.relax.in > polyene.relax.out
!grep "JOB DONE" polyene.relax.out

Now let's see what we got.

In [ ]:
!grep -A 7 "Begin final coordinates" polyene.relax.out

## 2. Now you are on your own.

1. Visualise the relaxed geometry using Avogadro and Vesta. What has changed?
2. Prepare and run ```relaxed.scf.in``` and ```relaxed.nscf.in```.
3. Visualise the band structure and density of states.
4. Compare the band structure and density of states of the cumulene and the relaxed geometry. Discuss.

## 3. Su–Schrieffer–Heeger (SSH) model

The SSH Hamiltonian can be written as:

$H = -(t + \Delta) \sum_{n\in\mathrm{even}}(c_n^\dagger c_{n+1} + \mathrm{h.c.}) -(t - \Delta) \sum_{n\in\mathrm{odd}}(c_n^\dagger c_{n+1} + \mathrm{h.c.})$

which gives the eigenvalues:

$E = \pm \sqrt{(t + \Delta)^2 + (t - \Delta)^2 + 2(t + \Delta)(t - \Delta)\cos(k)}$

The cumulenic, metallic geometry has:

$\Delta = 0$

while for the polyenic (relaxed) semiconductor we have:

$\Delta \neq 0$

Determine the values of $t$ and $\Delta$ from DFT!

In [ ]:
"""
Here are a few hints.
1. Determining delta is easier. Do that first.
2. Determining t is harder. It might help to:
 - limit yourself to k >= 0.3
 - zero the Fermi energy
3. Define two helper functions:
 - calc_tb_evs, which calculates the tight-binding eigenvalues for a given t and k
 - calc_ssq, which calculates the sum of squares between the tight-binding and DFT eigenvalues
4. Use the minimize function from scipy to find the t that minimizes the sum of squares.
"""

from scipy.optimize import minimize
t_guess = 2.0
# your code here
solution = minimize(calc_ssq, x0=t_guess, args=(dft_bands, k_points)) # fill in the arguments
print(f"Solution is {solution.x[0]}.")